## API 

In [2]:
import requests

In [3]:
title = "Уравнение_Ван-дер-Ваальса"
url = f"https://ru.wikipedia.org/api/rest_v1/page/html/{title}"
response = requests.get(url)
response

<Response [200]>

In [6]:
print(*response.headers.items(), sep="\n")

('cache-control', 'max-age=5')
('content-language', 'ru')
('content-type', 'text/html; charset=utf-8; profile="https://www.mediawiki.org/wiki/Specs/HTML/2.8.0"')
('etag', 'W/"151838280/4eaff13a-1327-11f1-9695-9fc0f3dab102/view/html+lang:en-gb"')
('last-modified', 'Thu, 26 Feb 2026 12:44:47 GMT')


In [9]:
%pip install lxml

In [36]:
import pandas as pd
tables = pd.read_html(response.text, decimal=",", thousands=None)

<ipython-input-36-243f8655a0a6>:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, decimal=",", thousands=None)


In [11]:
len(tables) == 3

True

In [37]:
df = tables[0]
df.head(2)

,Вещество,"a, Па·м6·моль−2","b, 10−6 м3·моль−1"
0,Азот N2,0.1370,38.7
1,Аммиак NH3,0.4225,37.1


In [23]:
df["Вещество"].str.rsplit(" ", n=1, expand=True) \
    .rename(columns={0: "Name", 1: "Formula"}) \
    .head(1)

,Name,Formula
0,Азот,N2


In [38]:
df[["Name", "Formula"]] = \
    df["Вещество"].str.rsplit(" ", n=1, expand=True)

In [26]:
df.head(1)

,Вещество,"a, Па·м6·моль−2","b, 10−6 м3·моль−1",Name,Formula
0,Азот N2,1370,387,Азот,N2


In [31]:
list(df.columns)

['Вещество', 'a,  Па·м6·моль−2', 'b,  10−6 м3·моль−1', 'Name', 'Formula']

In [39]:
df_pretty = df[
        ["Name", "Formula", "a,  Па·м6·моль−2", "b,  10−6 м3·моль−1"]
    ] \
    .rename(columns={
        "a,  Па·м6·моль−2": "A", "b,  10−6 м3·моль−1": "B"
    })

df_pretty.head(1)

,Name,Formula,A,B
0,Азот,N2,0.137,38.7


## Pubchempy

In [40]:
%pip install ssl pyodide-http pubchempy

In [41]:
import pyodide_http
pyodide_http.patch_all()

In [42]:
import pubchempy

In [44]:
h2_compounds = pubchempy.get_compounds("H2", namespace="formula")
h2_compounds

[Compound(783),
 Compound(24523),
 Compound(24824),
 Compound(167583),
 Compound(5460631),
 Compound(119434),
 Compound(6914304),
 Compound(6914290),
 Compound(157679700),
 Compound(159167674),
 Compound(169430093)]

In [52]:
h2_compound_0 = h2_compounds[0]
h2_compound_0.molecular_weight

2.016

In [73]:
import time
t0 = time.time()

df_with_mw = df_pretty.sort_values("B").head(3)

df_with_mw["molecular_weight"] = df_with_mw["Formula"] \
    .map(lambda f: pubchempy.get_compounds(f, namespace="formula")[0].molecular_weight)

print(f"{time.time() - t0:.1f}", "s")
df_with_mw

<class 'KeyError'>: 'PC_Compounds'

In [71]:
t0

1773657580.708

In [77]:
df_pretty.to_csv(
    "./Уравнение_Ван-дер-Ваальса.tsv",
    sep="\t",
    index=False
)